# CIFAR - 10 Object Recognition using ResNet50

To run notebook download data from "https://www.kaggle.com/competitions/cifar-10/data" and unpack as is in the folder with notebook. 

## Extract Dataset & Organize into Class Subfolders

In [4]:
import os
import shutil
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from keras.src.legacy.preprocessing.image import ImageDataGenerator

In [3]:
train_dir = './train'
labels = pd.read_csv('./trainLabels.csv')

# Create subfolders for each class
for class_name in labels['label'].unique():
    os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)

# Move images into class subfolders
for _, row in labels.iterrows():
    filename = str(row['id']) + '.png'
    src = os.path.join(train_dir, filename)
    dst = os.path.join(train_dir, row['label'], filename)
    if os.path.exists(src):
        shutil.move(src, dst)

# Print count per class
for class_name in sorted(labels['label'].unique()):
    count = len(os.listdir(os.path.join(train_dir, class_name)))
    print(f'{class_name}: {count}')

airplane: 5000
automobile: 5000
bird: 5000
cat: 5000
deer: 5000
dog: 5000
frog: 5000
horse: 5000
ship: 5000
truck: 5000


## Data Loading & Preprocessing (ImageDataGenerator)

In [5]:
# Training generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2
)

# Training set
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

# Validation set (no augmentation, just rescale)
validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 40000 images belonging to 10 classes.
Found 10000 images belonging to 10 classes.


## Load Pre-trained ResNet50 & Build Custom Head

In [6]:
from  keras.applications import ResNet50

In [8]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 47s 0us/step


In [9]:
for layer in base_model.layers:
    layer.trainable = False

In [10]:
model = keras.Sequential([
    base_model,
    keras.layers.Flatten(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

##  Compile & Train the Model

In [11]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,280,650 (187.99 MB)

 Trainable params: 25,692,938 (98.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
history = model.fit(train_generator, validation_data=validation_generator, epochs=1)

 508/1250 ━━━━━━━━━━━━━━━━━━━━ 11:42 947ms/step - accuracy: 0.1019 - loss: 2.3053

## Evaluate & Visualize Results

In [ ]:
score, acc = model.evaluate(validation_generator)
print('Test Loss =', score)
print('Test Accuracy =', acc)

In [ ]:
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Training', 'Validation'], loc='lower right')
plt.show()

In [ ]:
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['Training', 'Validation'], loc='upper right')
plt.show()

## Prediction system

In [ ]:
# Class names
class_names = list(train_generator.class_indices.keys())

# Load and predict a single image
img_path = './train/frog/1.png'
img = keras.utils.load_img(img_path, target_size=(224, 224))
img_array = keras.utils.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
predicted_class = class_names[np.argmax(prediction)]
confidence = np.max(prediction)

print('Prediction:', predicted_class)
print('Confidence:', confidence)

plt.imshow(keras.utils.load_img(img_path))
plt.title(predicted_class)
plt.axis('off')
plt.show()
